# Naive Bayes Baseline: MultinomialNB + TF-IDF

Classical ML baseline for comparison with BERT-based classifiers.
Single-label classification (13 classes) on German news articles (2025 Bundestagswahl).

**Pipeline:**
1. 3-Fold Stratified Cross-Validation (identical splits to EuroBERT-210m baseline)
2. Final model trained on full CV pool, evaluated on frozen test set
3. Report generation (same format as BERT reports)

**Prerequisites:** No GPU needed. CPU runtime is sufficient.

In [ ]:
# === SETUP ===
import os, sys

# Clone / update repo
REPO = "/content/news_articles_classification_thesis"
if not os.path.exists(REPO):
    !git clone https://github.com/ZorbeyOezcan/news_articles_classification_thesis.git {REPO}
else:
    !cd {REPO} && git pull -q

# Dependencies (no torch/transformers needed)
!pip install -q scikit-learn datasets huggingface_hub matplotlib seaborn pandas tqdm

# Mount Google Drive (persistent reports)
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# Make pipeline_utils importable
PIPELINE_DIR = f"{REPO}/Python/classification_pipeline/euroBERT_210m"
if PIPELINE_DIR not in sys.path:
    sys.path.insert(0, PIPELINE_DIR)

import importlib
import pipeline_utils as pu
importlib.reload(pu)

# Create output directories on Google Drive
pu.ensure_drive_dirs()

print(f"Reports directory: {pu.REPORTS_DIR}")
print("Setup complete.")

In [ ]:
# ===== CONFIGURATION =====
import numpy as np

MODEL_SHORT_NAME = "naive_bayes_multinomial"
MODEL_TYPE = "traditional-ml"

# ----- Cross-validation & split (identical to BERT baseline) -----
N_FOLDS = 3
TEST_PER_CLASS = 30
VAL_FRACTION = 0.2
RANDOM_SEED = 42

# ----- TF-IDF configuration -----
TFIDF_MAX_FEATURES = 50_000
TFIDF_NGRAM_RANGE = (1, 2)     # unigrams + bigrams
TFIDF_SUBLINEAR_TF = True      # apply log normalization
TFIDF_MIN_DF = 2               # ignore terms appearing in <2 docs

# ----- Naive Bayes configuration -----
NB_ALPHA = 1.0  # Laplace smoothing (default)

# ----- Label list (must match dataset exactly) -----
ALL_LABELS = [
    "Klima / Energie", "Zuwanderung", "Renten", "Soziales Gefälle",
    "AfD/Rechte", "Arbeitslosigkeit", "Wirtschaftslage", "Politikverdruss",
    "Gesundheitswesen, Pflege", "Kosten/Löhne/Preise",
    "Ukraine/Krieg/Russland", "Bundeswehr/Verteidigung", "Andere",
]

label2id = {label: idx for idx, label in enumerate(ALL_LABELS)}
id2label = {idx: label for idx, label in enumerate(ALL_LABELS)}

# ----- Model info (for report) -----
MODEL_INFO = {
    "huggingface_id": "N/A (sklearn)",
    "language": "Language-agnostic (bag-of-words)",
    "max_tokens": "N/A",
    "parameters": "N/A (non-parametric in DL sense)",
    "notes": (
        "MultinomialNB with TF-IDF features. Classical ML baseline for comparison "
        "with neural approaches."
    ),
}

print(f"TF-IDF: max_features={TFIDF_MAX_FEATURES}, ngrams={TFIDF_NGRAM_RANGE}, "
      f"sublinear_tf={TFIDF_SUBLINEAR_TF}, min_df={TFIDF_MIN_DF}")
print(f"NB alpha (smoothing): {NB_ALPHA}")
print(f"CV Folds: {N_FOLDS}")
print(f"Labels: {len(ALL_LABELS)} classes")

In [ ]:
# ===== LOAD DATA & CREATE SPLITS =====
# Identical split logic to 01_baseline_finetune.ipynb (cell-3)
# to ensure the exact same test set and CV folds.

import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split

np.random.seed(RANDOM_SEED)

ds = load_dataset(pu.DATASET_ID)
train_hf = ds["train"].to_pandas()
test_hf = ds["test"].to_pandas()
all_labelled = pd.concat([train_hf, test_hf], ignore_index=True)

print(f"Total pool of labelled articles: {len(all_labelled)}")
print(f"Classes: {all_labelled['label'].nunique()}\n")

# --- Step 1: Fixed stratified test split (identical to BERT baseline) ---
test_indices = []
rest_indices = []

for label in ALL_LABELS:
    label_mask = all_labelled["label"] == label
    label_indices = all_labelled[label_mask].index.tolist()
    n_total = len(label_indices)
    if n_total < 60:
        n_test = n_total // 2
        print(f"  {label}: only {n_total} articles -> {n_test} for test")
    else:
        n_test = TEST_PER_CLASS
    np.random.shuffle(label_indices)
    test_indices.extend(label_indices[:n_test])
    rest_indices.extend(label_indices[n_test:])

test_df = all_labelled.loc[test_indices].reset_index(drop=True)
cv_pool_df = all_labelled.loc[rest_indices].reset_index(drop=True)

print(f"\nTest split:  {len(test_df)} articles (frozen)")
print(f"CV pool:     {len(cv_pool_df)} articles (for 3-fold CV + final model)")

# --- Step 2: Label encoding ---
cv_pool_df["label_id"] = cv_pool_df["label"].map(label2id)
test_df["label_id"] = test_df["label"].map(label2id)

for name, _df in [("CV Pool", cv_pool_df), ("Test", test_df)]:
    assert _df["label_id"].isna().sum() == 0, f"Unknown labels in {name}!"

# --- Split config for report ---
split_config = {
    "dataset_id": pu.DATASET_ID,
    "split_mode": "custom_finetune_cv",
    "test_per_class": TEST_PER_CLASS,
    "val_fraction": VAL_FRACTION,
    "n_folds": N_FOLDS,
    "random_seed": RANDOM_SEED,
    "cv_pool_size": len(cv_pool_df),
    "train_size": len(cv_pool_df),
    "eval_size": 0,
    "test_size": len(test_df),
    "raw_size": 0,
}

# Class distribution overview
split_overview = pd.DataFrame({
    "CV Pool": cv_pool_df["label"].value_counts(),
    "Test": test_df["label"].value_counts(),
}).fillna(0).astype(int)
split_overview["Total"] = split_overview.sum(axis=1)
split_overview.loc["TOTAL"] = split_overview.sum()
print("\nClass distribution:")
print(split_overview.to_string())

In [ ]:
# ===== 3-FOLD STRATIFIED CROSS-VALIDATION =====
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

# Pre-compute fold indices (identical to BERT baseline)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
fold_indices = list(skf.split(cv_pool_df, cv_pool_df["label_id"]))

print(f"Stratified {N_FOLDS}-fold CV on {len(cv_pool_df)} articles:")
for i, (train_idx, val_idx) in enumerate(fold_indices):
    val_labels = cv_pool_df.iloc[val_idx]["label"].value_counts()
    print(f"  Fold {i+1}: Train={len(train_idx)}, Val={len(val_idx)}, "
          f"smallest val class: {val_labels.idxmin()} ({val_labels.min()} samples)")

# --- CV fold loop ---
fold_results = []
cv_timer = pu.ExperimentTimer()

with cv_timer:
    for fold_idx, (train_idx, val_idx) in enumerate(fold_indices):
        print(f"\n{'='*60}")
        print(f"  FOLD {fold_idx + 1}/{N_FOLDS}")
        print(f"{'='*60}")

        # Extract fold data
        fold_train_df = cv_pool_df.iloc[train_idx]
        fold_val_df = cv_pool_df.iloc[val_idx]

        # Fit TF-IDF on fold training data only (no data leakage)
        tfidf = TfidfVectorizer(
            max_features=TFIDF_MAX_FEATURES,
            ngram_range=TFIDF_NGRAM_RANGE,
            sublinear_tf=TFIDF_SUBLINEAR_TF,
            min_df=TFIDF_MIN_DF,
        )
        X_train = tfidf.fit_transform(fold_train_df["text"])
        X_val = tfidf.transform(fold_val_df["text"])
        y_train = fold_train_df["label_id"].values
        y_val = fold_val_df["label_id"].values

        print(f"  TF-IDF features: {X_train.shape[1]}")

        # Train MultinomialNB
        clf = MultinomialNB(alpha=NB_ALPHA)
        clf.fit(X_train, y_train)

        # Predict & evaluate
        y_pred = clf.predict(X_val)

        fold_f1 = f1_score(y_val, y_pred, average="macro", zero_division=0)
        fold_acc = accuracy_score(y_val, y_pred)
        fold_prec = precision_score(y_val, y_pred, average="macro", zero_division=0)
        fold_rec = recall_score(y_val, y_pred, average="macro", zero_division=0)

        fold_results.append({
            "fold": fold_idx + 1,
            "f1_macro": fold_f1,
            "accuracy": fold_acc,
            "precision_macro": fold_prec,
            "recall_macro": fold_rec,
        })
        print(f"  Fold {fold_idx + 1}: F1 Macro = {fold_f1:.4f}, "
              f"Acc = {fold_acc:.4f}, Prec = {fold_prec:.4f}, Rec = {fold_rec:.4f}")

print(f"\nCV complete. Duration: {cv_timer.duration_formatted}")

In [ ]:
# ===== CV RESULTS SUMMARY =====
import matplotlib.pyplot as plt

cv_metrics = {
    "f1_macro":        [r["f1_macro"] for r in fold_results],
    "accuracy":        [r["accuracy"] for r in fold_results],
    "precision_macro": [r["precision_macro"] for r in fold_results],
    "recall_macro":    [r["recall_macro"] for r in fold_results],
}

print(f"\n{'='*60}")
print(f"  NAIVE BAYES 3-FOLD CV RESULTS ({len(fold_results)}/{N_FOLDS} folds)")
print(f"{'='*60}")
print(f"{'Metric':<20} {'Mean':>8} {'Std':>8} {'Min':>8} {'Max':>8}")
print("-" * 55)
for metric_name, values in cv_metrics.items():
    print(f"{metric_name:<20} {np.mean(values):>8.4f} {np.std(values):>8.4f} "
          f"{np.min(values):>8.4f} {np.max(values):>8.4f}")
print(f"{'='*60}")

# Per-fold table
print(f"\nPer-fold breakdown:")
print(f"{'Fold':>6} {'F1 Macro':>10} {'Accuracy':>10} {'Precision':>10} {'Recall':>10}")
print("-" * 50)
for r in fold_results:
    print(f"{r['fold']:>6} {r['f1_macro']:>10.4f} {r['accuracy']:>10.4f} "
          f"{r['precision_macro']:>10.4f} {r['recall_macro']:>10.4f}")

# Store CV summary for report
cv_summary = {
    "n_folds": N_FOLDS,
    "folds_completed": len(fold_results),
    "mean_f1_macro": round(float(np.mean(cv_metrics["f1_macro"])), 4),
    "std_f1_macro": round(float(np.std(cv_metrics["f1_macro"])), 4),
    "mean_accuracy": round(float(np.mean(cv_metrics["accuracy"])), 4),
    "std_accuracy": round(float(np.std(cv_metrics["accuracy"])), 4),
    "mean_precision_macro": round(float(np.mean(cv_metrics["precision_macro"])), 4),
    "std_precision_macro": round(float(np.std(cv_metrics["precision_macro"])), 4),
    "mean_recall_macro": round(float(np.mean(cv_metrics["recall_macro"])), 4),
    "std_recall_macro": round(float(np.std(cv_metrics["recall_macro"])), 4),
    "per_fold_f1": [round(f, 4) for f in cv_metrics["f1_macro"]],
}

# Bar chart: fold comparison
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(fold_results))
width = 0.2

ax.bar(x - 1.5*width, [r["f1_macro"] for r in fold_results], width, label="F1 Macro", color="#4CAF50")
ax.bar(x - 0.5*width, [r["accuracy"] for r in fold_results], width, label="Accuracy", color="#2196F3")
ax.bar(x + 0.5*width, [r["precision_macro"] for r in fold_results], width, label="Precision", color="#FF9800")
ax.bar(x + 1.5*width, [r["recall_macro"] for r in fold_results], width, label="Recall", color="#9C27B0")

ax.set_xticks(x)
ax.set_xticklabels([f"Fold {r['fold']}" for r in fold_results])
ax.set_ylabel("Score")
ax.set_title("Naive Bayes 3-Fold CV: Per-Fold Metrics", fontsize=13, fontweight="bold")
ax.legend()
ax.set_ylim(0, 1)
ax.grid(axis="y", alpha=0.3)

# Add mean line
mean_f1 = np.mean(cv_metrics["f1_macro"])
ax.axhline(y=mean_f1, color="red", linestyle="--", alpha=0.7, label=f"Mean F1={mean_f1:.4f}")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ===== FINAL MODEL: TRAIN ON CV POOL, TEST ON FIXED TEST SET =====
timer = pu.ExperimentTimer()

with timer:
    # Fit TF-IDF on full CV pool
    tfidf_final = TfidfVectorizer(
        max_features=TFIDF_MAX_FEATURES,
        ngram_range=TFIDF_NGRAM_RANGE,
        sublinear_tf=TFIDF_SUBLINEAR_TF,
        min_df=TFIDF_MIN_DF,
    )
    X_train_final = tfidf_final.fit_transform(cv_pool_df["text"])
    X_test = tfidf_final.transform(test_df["text"])
    y_train_final = cv_pool_df["label_id"].values

    print(f"TF-IDF features (final): {X_train_final.shape[1]}")
    print(f"Training samples: {X_train_final.shape[0]}")
    print(f"Test samples: {X_test.shape[0]}")

    # Train final MultinomialNB
    clf_final = MultinomialNB(alpha=NB_ALPHA)
    clf_final.fit(X_train_final, y_train_final)

    # Predict on test set
    y_test_pred_ids = clf_final.predict(X_test)

# Convert to label strings for pu.evaluate()
pred_labels = [id2label[i] for i in y_test_pred_ids]
true_labels = test_df["label"].tolist()

# Compute metrics using pipeline_utils (identical metric computation as BERT)
metrics = pu.evaluate(
    true_labels, pred_labels,
    labels=ALL_LABELS, experiment_name="test",
)
pu.print_metrics(metrics, "Naive Bayes MultinomialNB -- Test Split")
print(f"Duration: {timer.duration_formatted}")

In [ ]:
# ===== CONFUSION MATRIX =====
pu.plot_confusion_matrix(metrics, title="Naive Bayes MultinomialNB (Test)")

In [ ]:
# ===== PER-CLASS METRICS BARPLOT =====
pc_df = metrics["per_class_df"].copy().sort_values("F1", ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))
y_pos = np.arange(len(pc_df))
bar_h = 0.25

ax.barh(y_pos - bar_h, pc_df["Precision"], bar_h, label="Precision", color="#2196F3", alpha=0.85)
ax.barh(y_pos, pc_df["Recall"], bar_h, label="Recall", color="#FF9800", alpha=0.85)
ax.barh(y_pos + bar_h, pc_df["F1"], bar_h, label="F1", color="#4CAF50", alpha=0.85)

ax.set_yticks(y_pos)
ax.set_yticklabels(pc_df["Label"])
ax.set_xlabel("Score")
ax.set_title("Per-Class Metrics: Naive Bayes MultinomialNB (Test)", fontsize=13, fontweight="bold")
ax.legend(loc="lower right")
ax.set_xlim(0, 1.05)
ax.grid(axis="x", alpha=0.3)

for i, (_, row) in enumerate(pc_df.iterrows()):
    ax.text(row["F1"] + 0.01, y_pos[i] + bar_h, f"{row['F1']:.2f}", va="center", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ===== GENERATE REPORT =====
training_params = {
    "source": "Naive Bayes baseline (MultinomialNB + TF-IDF)",
    "classifier": "sklearn.naive_bayes.MultinomialNB",
    "alpha (smoothing)": NB_ALPHA,
    "tfidf_max_features": TFIDF_MAX_FEATURES,
    "tfidf_ngram_range": str(TFIDF_NGRAM_RANGE),
    "tfidf_sublinear_tf": TFIDF_SUBLINEAR_TF,
    "tfidf_min_df": TFIDF_MIN_DF,
    "tfidf_features_final": X_train_final.shape[1],
    "training_samples": X_train_final.shape[0],
    "test_samples": X_test.shape[0],
    # CV results
    "cv_n_folds": cv_summary["n_folds"],
    "cv_folds_completed": cv_summary["folds_completed"],
    "cv_mean_f1_macro": cv_summary["mean_f1_macro"],
    "cv_std_f1_macro": cv_summary["std_f1_macro"],
    "cv_mean_accuracy": cv_summary["mean_accuracy"],
    "cv_std_accuracy": cv_summary["std_accuracy"],
    "cv_per_fold_f1": cv_summary["per_fold_f1"],
}

report_path = pu.generate_report(
    model_name=MODEL_SHORT_NAME,
    model_type=MODEL_TYPE,
    metrics=metrics,
    timer=timer,
    model_info=MODEL_INFO,
    candidate_labels=ALL_LABELS,
    split_config=split_config,
    label_mapping={l: l for l in ALL_LABELS},
    training_params=training_params,
    experiment_notes=(
        f"BASELINE: MultinomialNB with TF-IDF features (classical ML baseline). "
        f"3-fold stratified CV: F1 Macro = {cv_summary['mean_f1_macro']:.4f} "
        f"+/- {cv_summary['std_f1_macro']:.4f}. "
        f"Final model trained on {len(cv_pool_df)} articles (full CV pool), "
        f"tested on {len(test_df)} articles (frozen test set). "
        f"TF-IDF: max_features={TFIDF_MAX_FEATURES}, ngrams={TFIDF_NGRAM_RANGE}, "
        f"sublinear_tf={TFIDF_SUBLINEAR_TF}, min_df={TFIDF_MIN_DF}. "
        f"NB alpha={NB_ALPHA}."
    ),
)
print(f"Report saved: {report_path}")

In [ ]:
# ===== SUMMARY =====
print("=" * 70)
print(f"  NAIVE BAYES BASELINE COMPLETE")
print("=" * 70)
print(f"  Classifier:      MultinomialNB (alpha={NB_ALPHA})")
print(f"  Features:        TF-IDF (max={TFIDF_MAX_FEATURES}, ngrams={TFIDF_NGRAM_RANGE})")
print(f"")
print(f"  --- 3-Fold CV Results ---")
print(f"  CV F1 Macro:     {cv_summary['mean_f1_macro']:.4f} +/- {cv_summary['std_f1_macro']:.4f}")
print(f"  CV Accuracy:     {cv_summary['mean_accuracy']:.4f} +/- {cv_summary['std_accuracy']:.4f}")
print(f"  CV Precision:    {cv_summary['mean_precision_macro']:.4f} +/- {cv_summary['std_precision_macro']:.4f}")
print(f"  CV Recall:       {cv_summary['mean_recall_macro']:.4f} +/- {cv_summary['std_recall_macro']:.4f}")
print(f"  Per-fold F1:     {cv_summary['per_fold_f1']}")
print(f"  CV Duration:     {cv_timer.duration_formatted}")
print(f"")
print(f"  --- Final Model (Test Set) ---")
print(f"  Train:           {len(cv_pool_df)} articles (full CV pool)")
print(f"  Test:            {len(test_df)} articles (frozen)")
print(f"  F1 Macro:        {metrics['f1_macro']:.4f}")
print(f"  F1 Weighted:     {metrics['f1_weighted']:.4f}")
print(f"  Accuracy:        {metrics['accuracy']:.4f}")
print(f"  Precision Macro: {metrics['precision_macro']:.4f}")
print(f"  Recall Macro:    {metrics['recall_macro']:.4f}")
print(f"  Duration:        {timer.duration_formatted}")
print(f"")
print(f"  Report:          {report_path}")
print("=" * 70)
print("\nDone. Runtime can be terminated.")